# ASDiv — CoT Baseline Evaluation
## Chain-of-Thought vs No-Guidance Baseline

**Purpose:** Runs TWO conditions on the same 300 ASDiv questions (seed=42):
- **CoT:** Qwen 1.5B × 5 votes with `Let's think step by step`
- **Baseline:** Qwen 1.5B × 5 votes, no CoT, no guide

Compare against **Guided pipeline** results from the original notebook.

| Condition | Compute | Description |
|---|---|---|
| Baseline | 7.5B | No guide, no CoT |
| **CoT (this notebook)** | **7.5B** | No guide, 'think step by step' |
| Guided (original notebook) | 10.5B | Fine-tuned 3B guide + 1.5B solver |

**Same seed=42, same 300 questions — direct 3-way comparison is valid.**

In [1]:
# CELL 1 -- Install
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [2]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login

login("os.environ["HF_TOKEN"]")  # paste your token here
print("HuggingFace login done")

HuggingFace login done


In [3]:
# CELL 3 -- Imports + GPU check
import os, json, re, random, time
import torch
import numpy as np
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/asdiv_cot_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")

PyTorch : 2.9.0+cu126
GPU     : Tesla P100-PCIE-16GB
VRAM    : 17.1 GB
Output  : /kaggle/working/asdiv_cot_eval


In [4]:
# CELL 4 -- Configuration
# NOTE: No guide model. Only the 1.5B solver.
# CRITICAL: max_eval_samples=300 and random_seed=42 MUST match original notebook.
CONFIG = {
    "solver_model"     : "Qwen/Qwen2.5-1.5B-Instruct",
    "dataset_name"     : "EleutherAI/asdiv",
    "dataset_split"    : "validation",
    "max_eval_samples" : 300,   # MUST match original notebook
    "random_seed"      : 42,    # MUST match original notebook
    "n_votes"          : 5,
    "vote_temperature" : 0.4,   # MUST match original notebook
    "max_new_tokens"   : 350,
    "solver_params_B"  : 1.5,
    "results_file"     : f"{OUTPUT_DIR}/results.jsonl",
    "angle1_file"      : f"{OUTPUT_DIR}/angle1_compute.json",
    "angle2_file"      : f"{OUTPUT_DIR}/angle2_consistency.json",
    "angle3_file"      : f"{OUTPUT_DIR}/angle3_calibration.json",
    "angle4_file"      : f"{OUTPUT_DIR}/angle4_by_optype.json",
    "checkpoint_file"  : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"       : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")

Config ready:
  solver_model            : Qwen/Qwen2.5-1.5B-Instruct
  dataset_name            : EleutherAI/asdiv
  dataset_split           : validation
  max_eval_samples        : 300
  random_seed             : 42
  n_votes                 : 5
  vote_temperature        : 0.4
  max_new_tokens          : 350
  solver_params_B         : 1.5
  results_file            : /kaggle/working/asdiv_cot_eval/results.jsonl
  angle1_file             : /kaggle/working/asdiv_cot_eval/angle1_compute.json
  angle2_file             : /kaggle/working/asdiv_cot_eval/angle2_consistency.json
  angle3_file             : /kaggle/working/asdiv_cot_eval/angle3_calibration.json
  angle4_file             : /kaggle/working/asdiv_cot_eval/angle4_by_optype.json
  checkpoint_file         : /kaggle/working/asdiv_cot_eval/checkpoint.json
  save_every              : 25


In [5]:
# CELL 5 -- Load ASDiv dataset
# Same seed=42 and max_eval_samples=300 guarantees the same 300 questions as original.

print("Loading ASDiv...")
raw_ds = load_dataset(CONFIG["dataset_name"])
print(f"Split size: {len(raw_ds[CONFIG['dataset_split']])}")

OP_MAP = {
    "addition": "Addition", "sum": "Addition",
    "subtraction": "Subtraction", "difference": "Subtraction", "comparison": "Subtraction",
    "multiplication": "Multiplication",
    "division": "Division", "common-division": "Division", "floor-division": "Division",
}

def broad_op(solution_type):
    st = str(solution_type).lower().strip()
    for key, val in OP_MAP.items():
        if key in st: return val
    return "Multi-step"

def normalise_asdiv(item):
    q = item["body"].strip().rstrip(".") + " " + item["question"].strip()
    raw_ans = str(item["answer"]).strip().replace(",", "")
    m = re.match(r"(-?[\d\.]+)", raw_ans)
    ans_str = m.group(1) if m else raw_ans
    try:
        f = float(ans_str)
        ans_str = str(int(f)) if f == int(f) else str(round(f, 4))
    except Exception: pass
    return {
        "question": q,
        "answer": ans_str,
        "op_type": broad_op(item.get("solution_type", "")),
        "solution_type": str(item.get("solution_type", "")),
    }

all_data = [normalise_asdiv(x) for x in raw_ds[CONFIG["dataset_split"]]]

# Fix seed ONCE -- must match original notebook
random.seed(CONFIG["random_seed"])
test_data = random.sample(all_data, CONFIG["max_eval_samples"])

op_counts = Counter(d["op_type"] for d in test_data)
print(f"Sampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
for op, cnt in sorted(op_counts.items(), key=lambda x: -x[1]):
    print(f"  {op:<20}: {cnt}")
print(f"First Q : {test_data[0]['question'][:80]}")
print(f"First A : {test_data[0]['answer']}  |  op: {test_data[0]['op_type']}")

Loading ASDiv...


README.md:   0%|          | 0.00/494 [00:00<?, ?B/s]

asdiv/validation-00000-of-00001.parquet:   0%|          | 0.00/267k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/2305 [00:00<?, ? examples/s]

Split size: 2305
Sampled 300 questions (seed=42)
  Multi-step          : 89
  Subtraction         : 67
  Addition            : 61
  Multiplication      : 43
  Division            : 40
First Q : He also has a section filled with short story booklets. If each booklet has 9 pa
First A : 441  |  op: Multiplication


In [6]:
# CELL 6 -- Answer extraction (identical to original notebook)

def normalise_num(s):
    s = s.replace(",", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except ValueError: return s

def extract_gt_answer(answer_str):
    return normalise_num(str(answer_str))

def extract_pred_answer(text):
    m = re.search(r"####\s*(-?[\d\.]+)", text)
    if m: return normalise_num(m.group(1))
    m = re.search(r"\\boxed\{(-?[\d\.]+)\}", text)
    if m: return normalise_num(m.group(1))
    m = re.search(r"(?:the answer is|answer is)\s*:?\s*\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    m = re.search(r"=\s*\$?(-?[\d\.]+)\s*$", text.strip(), re.MULTILINE)
    if m: return normalise_num(m.group(1))
    m = re.search(r"\*\*\$?(-?[\d\.]+)\*\*\.?\s*$", text.strip())
    if m: return normalise_num(m.group(1))
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    return ""

# Self-test
_tests = [
    ("#### 42", "42"), ("#### 3.5", "3.5"), ("\\boxed{100}", "100"),
    ("The answer is 7", "7"), ("Total = 20", "20"),
    ("**200**.", "200"), ("Therefore, 13", "13"), ("Some text", ""),
]
all_ok = all(extract_pred_answer(t) == e for t, e in _tests)
print("Extractor:", "ALL PASSED" if all_ok else "FAILURES -- fix before running eval")

Extractor: ALL PASSED


In [7]:
# CELL 7 -- Load solver model (Qwen 1.5B only -- no guide model needed)
# T4 has 15GB. 1.5B in float16 ~ 3GB. Plenty of headroom.

print(f"Loading: {CONFIG['solver_model']}")
solver_tok = AutoTokenizer.from_pretrained(CONFIG["solver_model"])
if solver_tok.pad_token is None:
    solver_tok.pad_token = solver_tok.eos_token

solver_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["solver_model"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

vram = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used : {vram:.2f} GB")
print(f"Headroom  : {15.0 - vram:.1f} GB")
print("Solver ready")

Loading: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

VRAM used : 3.09 GB
Headroom  : 11.9 GB
Solver ready


In [8]:
# CELL 8 -- Prompts and generation functions
#
# COT_SYSTEM      -- adds 'Let's think step by step' (the new CoT condition)
# BASELINE_SYSTEM -- no CoT, no guide  (matches original notebook baseline)
# REFINER_SYSTEM  -- tie-breaker (same as original)

COT_SYSTEM = (
    "You are a precise math problem solver.\n"
    "Let's think step by step.\n"
    "Show every arithmetic calculation clearly.\n"
    "Your FINAL line must be exactly: #### [number]"
)

BASELINE_SYSTEM = (
    "You are a precise math problem solver.\n"
    "Read the problem carefully. Solve step by step, showing every calculation.\n"
    "Your FINAL line must be exactly: #### [number]"
)

REFINER_SYSTEM = (
    "You are a careful math problem solver.\n"
    "Previous attempts gave different answers. Ignore all of them.\n"
    "Re-solve completely from scratch. Show every arithmetic step.\n"
    "Your FINAL line must be exactly: #### [number]"
)


def run_solver(messages, temperature):
    prompt = solver_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = solver_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(solver_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = solver_model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=max(temperature, 0.05),
            do_sample=True,
            top_p=0.92,
            top_k=40,
            pad_token_id=solver_tok.eos_token_id,
            repetition_penalty=1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return solver_tok.decode(new_toks, skip_special_tokens=True).strip()


def generate_cot(question):
    """CoT: Let's think step by step — no guide."""
    return run_solver(
        [{"role": "system", "content": COT_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        temperature=CONFIG["vote_temperature"],
    )


def generate_baseline(question):
    """Baseline: no CoT, no guide."""
    return run_solver(
        [{"role": "system", "content": BASELINE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        temperature=CONFIG["vote_temperature"],
    )


def generate_refiner(question, candidates):
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"Problem: {question}\n\n"
        f"Previous attempts gave: {cands}\n"
        "Ignore all previous attempts. Solve from scratch:"
    )
    return run_solver(
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        temperature=0.3,
    )


print("Generation functions ready")
print("  CoT prompt   : 'Let's think step by step'")
print("  Temperature  :", CONFIG["vote_temperature"])

Generation functions ready
  CoT prompt   : 'Let's think step by step'
  Temperature  : 0.4


In [9]:
# CELL 9 -- Voting logic (identical to original notebook)

def vote_and_decide(answers, question, gt_answer=None):
    valid = [a for a in answers if a and a.strip()]
    if not valid: valid = answers

    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        ref_raw  = generate_refiner(question, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None
        all_v      = valid + ([ref_ans] if ref_ans else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]
        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted           = total - new_top_c
        vote_counts      = new_counts

    return {
        "final_answer": final, "strategy": strategy,
        "confidence": conf, "vote_counts": dict(vote_counts),
        "correct_votes": correct_votes, "total_votes": total,
        "vote_consistency": round(vote_consistency, 4),
        "wasted_votes": wasted,
        "refiner_used": refiner_used, "refiner_correct": refiner_correct,
    }

print("Voting logic ready")

Voting logic ready


In [10]:
# CELL 10 -- Single question test (verify both conditions work before full run)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
op   = item["op_type"]

print("=" * 65)
print(f"Question : {q}")
print(f"GT Answer: {gt}  |  op: {op}")

# CoT
print("\n[CoT] 3 sample votes:")
for i in range(3):
    raw  = generate_cot(q)
    pred = extract_pred_answer(raw)
    print(f"  Vote {i+1}: '{pred}'  | raw[:80]: {raw[:80]}")

# Baseline
print("\n[Baseline] 3 sample votes:")
for i in range(3):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    print(f"  Vote {i+1}: '{pred}'")

print("\nSingle test done. Run Cell 11 for full evaluation.")

Question : He also has a section filled with short story booklets. If each booklet has 9 pages and there are 49 booklets in the short story section, how many pages will Jack need to go through if he plans to read them all?
GT Answer: 441  |  op: Multiplication

[CoT] 3 sample votes:
  Vote 1: '441'  | raw[:80]: To calculate the total number of pages Jack needs to read:

1) First, we know th
  Vote 2: ''  | raw[:80]: To find out how many pages Jack needs to read, we multiply the number of pages p
  Vote 3: '441'  | raw[:80]: To find out how many pages Jack needs to go through, we multiply the number of p

[Baseline] 3 sample votes:
  Vote 1: '441'
  Vote 2: ''
  Vote 3: '441'

Single test done. Run Cell 11 for full evaluation.


In [ ]:
# CELL 11 -- Full Dual Evaluation Loop
# Runs all 300 questions under TWO conditions:
#   Mode A: cot      (1.5B x5, 'Let's think step by step')
#   Mode B: baseline (1.5B x5, no CoT)
# Checkpoints every 25 questions.

print(f"Dual evaluation: {len(test_data)} ASDiv questions")
print(f"Each question  : {CONFIG['n_votes']} CoT votes + {CONFIG['n_votes']} baseline votes")
print("-" * 65)

cot_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        cot_results  = [r for r in lines if r.get("mode") == "cot"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from {start_idx} (CoT: {len(cot_results)}, Base: {len(base_results)})")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="ASDiv CoT Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])
    op_type   = item["op_type"]

    # ---- CoT condition ----------------------------------------
    try:
        cot_votes_raw = [extract_pred_answer(generate_cot(question))
                         for _ in range(CONFIG["n_votes"])]
        c_dec = vote_and_decide(cot_votes_raw, question, gt_answer)
        cot_results.append({
            "mode": "cot", "idx": idx, "question": question,
            "gt_answer": gt_answer, "op_type": op_type,
            "final_answer": c_dec["final_answer"],
            "correct": c_dec["final_answer"] == gt_answer,
            "strategy": c_dec["strategy"], "confidence": c_dec["confidence"],
            "correct_votes": c_dec["correct_votes"], "total_votes": c_dec["total_votes"],
            "vote_consistency": c_dec["vote_consistency"],
            "wasted_votes": c_dec["wasted_votes"],
            "refiner_used": c_dec["refiner_used"],
            "refiner_correct": c_dec["refiner_correct"],
            "vote_counts": c_dec["vote_counts"],
        })
    except RuntimeError as e:
        cot_results.append({
            "mode": "cot", "idx": idx, "question": question,
            "gt_answer": gt_answer, "op_type": op_type,
            "final_answer": "", "correct": False, "strategy": "error",
            "confidence": 0.0, "correct_votes": 0,
            "total_votes": CONFIG["n_votes"], "vote_consistency": 0.0,
            "wasted_votes": CONFIG["n_votes"], "refiner_used": False,
            "refiner_correct": None, "vote_counts": {}, "error": str(e),
        })

    # ---- Baseline condition -----------------------------------
    try:
        base_votes_raw = [extract_pred_answer(generate_baseline(question))
                          for _ in range(CONFIG["n_votes"])]
        b_dec = vote_and_decide(base_votes_raw, question, gt_answer)
        base_results.append({
            "mode": "baseline", "idx": idx, "question": question,
            "gt_answer": gt_answer, "op_type": op_type,
            "final_answer": b_dec["final_answer"],
            "correct": b_dec["final_answer"] == gt_answer,
            "strategy": b_dec["strategy"], "confidence": b_dec["confidence"],
            "correct_votes": b_dec["correct_votes"], "total_votes": b_dec["total_votes"],
            "vote_consistency": b_dec["vote_consistency"],
            "wasted_votes": b_dec["wasted_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode": "baseline", "idx": idx, "question": question,
            "gt_answer": gt_answer, "op_type": op_type,
            "final_answer": "", "correct": False, "strategy": "error",
            "confidence": 0.0, "correct_votes": 0,
            "total_votes": CONFIG["n_votes"], "vote_consistency": 0.0,
            "wasted_votes": CONFIG["n_votes"], "refiner_used": False,
            "refiner_correct": None, "vote_counts": {}, "error": str(e),
        })

    # Checkpoint
    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in cot_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:3d}] CoT: {c_acc:.1f}%  Baseline: {b_acc:.1f}%  ({mins:.1f} min)")

# Final save
with open(CONFIG["results_file"], "w") as f:
    for r in cot_results + base_results:
        f.write(json.dumps(r) + "\n")

c_c = sum(r["correct"] for r in cot_results)
b_c = sum(r["correct"] for r in base_results)
print(f"\nDone. CoT: {c_c}/{len(cot_results)} = {c_c/len(cot_results)*100:.1f}%")
print(f"Baseline : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%")
print(f"CoT vs Baseline: {(c_c/len(cot_results) - b_c/len(base_results))*100:+.1f} pts") 

Dual evaluation: 300 ASDiv questions
Each question  : 5 CoT votes + 5 baseline votes
-----------------------------------------------------------------
Starting fresh


ASDiv CoT Eval:   0%|          | 0/300 [00:00<?, ?it/s]

  [ 25] CoT: 44.0%  Baseline: 52.0%  (28.3 min)
  [ 50] CoT: 44.0%  Baseline: 46.0%  (58.3 min)
  [ 75] CoT: 41.3%  Baseline: 41.3%  (87.6 min)
  [100] CoT: 42.0%  Baseline: 46.0%  (112.2 min)
  [125] CoT: 43.2%  Baseline: 45.6%  (136.2 min)
  [150] CoT: 45.3%  Baseline: 46.0%  (160.7 min)
  [175] CoT: 46.9%  Baseline: 48.0%  (185.2 min)
  [200] CoT: 46.0%  Baseline: 48.0%  (209.9 min)
  [225] CoT: 45.8%  Baseline: 48.4%  (231.9 min)
  [250] CoT: 45.2%  Baseline: 48.8%  (258.4 min)
  [275] CoT: 45.8%  Baseline: 47.6%  (286.7 min)


In [ ]:
# CELL 12 -- ANGLE 1: COMPUTE EFFICIENCY
# Both CoT and Baseline cost 7.5B (1.5B x5). Zero guide overhead.

S, N = CONFIG["solver_params_B"], CONFIG["n_votes"]
compute = S * N  # 7.5B for both

c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
c_wasted = sum(r["wasted_votes"] for r in cot_results)
b_wasted = sum(r["wasted_votes"] for r in base_results)
c_ref = sum(r["refiner_used"] for r in cot_results)
b_ref = sum(r["refiner_used"] for r in base_results)

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY")
print("=" * 65)
print(f"\n  {'Setup':<28} | {'Compute':>8} | {'Accuracy':>9}")
print(f"  {'-'*28}-+-{'-'*8}-+-{'-'*9}")
print(f"  {'Baseline (no CoT)':<28} | {compute:>6.1f}B  | {b_acc:>8.1f}%")
print(f"  {'CoT (think step by step)':<28} | {compute:>6.1f}B  | {c_acc:>8.1f}%")
print(f"\n  Accuracy gain (CoT vs Baseline): {c_acc - b_acc:+.1f} pts")
print(f"  Compute difference              : 0 (same 7.5B)")
print(f"  Wasted votes saved              : {b_wasted - c_wasted} ({c_wasted} CoT vs {b_wasted} Baseline)")
print(f"  Refiner: CoT={c_ref}  Baseline={b_ref}")

angle1 = {
    "dataset": "ASDiv",
    "n_questions": len(cot_results),
    "compute_B": compute,
    "cot_accuracy": round(c_acc, 2),
    "baseline_accuracy": round(b_acc, 2),
    "accuracy_gain": round(c_acc - b_acc, 2),
    "cot_wasted": c_wasted, "baseline_wasted": b_wasted,
    "cot_refiner": c_ref, "baseline_refiner": b_ref,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(angle1, f, indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")

In [ ]:
# CELL 13 -- ANGLE 2: VOTE CONSISTENCY

c_cons = [r["vote_consistency"] for r in cot_results]
b_cons = [r["vote_consistency"] for r in base_results]
c_mean = np.mean(c_cons)
b_mean = np.mean(b_cons)
lift   = c_mean / max(b_mean, 1e-6)

cot_wins      = sum(1 for c, b in zip(c_cons, b_cons) if c > b)
baseline_wins = sum(1 for c, b in zip(c_cons, b_cons) if b > c)
tied          = sum(1 for c, b in zip(c_cons, b_cons) if c == b)

def bucket(scores):
    return {
        "all_wrong (0%)":    sum(1 for s in scores if s == 0.0),
        "low (1-39%)":       sum(1 for s in scores if 0.0 < s < 0.4),
        "medium (40-79%)":   sum(1 for s in scores if 0.4 <= s < 0.8),
        "high (80-100%)":    sum(1 for s in scores if s >= 0.8),
    }

c_dist = bucket(c_cons)
b_dist = bucket(b_cons)

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY")
print("=" * 65)
print(f"\n  CoT mean consistency     : {c_mean*100:.1f}%")
print(f"  Baseline mean consistency: {b_mean*100:.1f}%")
print(f"  Consistency lift         : {lift:.2f}x")
print(f"\n  CoT wins / Baseline wins / Tied: {cot_wins} / {baseline_wins} / {tied}")
print(f"\n  {'Bucket':<22} | {'CoT':>8} | {'Baseline':>8} | {'Diff':>6}")
for bkt in ["all_wrong (0%)", "low (1-39%)", "medium (40-79%)", "high (80-100%)"]:
    cv, bv = c_dist[bkt], b_dist[bkt]
    print(f"  {bkt:<22} | {cv:>8} | {bv:>8} | {cv-bv:>+6}")

angle2 = {
    "cot_mean_consistency": round(c_mean, 4),
    "baseline_mean_consistency": round(b_mean, 4),
    "consistency_lift": round(lift, 4),
    "cot_wins": cot_wins, "baseline_wins": baseline_wins, "tied": tied,
    "cot_distribution": c_dist, "baseline_distribution": b_dist,
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(angle2, f, indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")

In [ ]:
# CELL 14 -- ANGLE 3: CONFIDENCE CALIBRATION

def calibration_report(results, label):
    buckets = [
        ("Very High (>=0.80)",    lambda c: c >= 0.80, 0.90),
        ("High (0.60-0.80)",      lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium (0.40-0.60)",    lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low (<0.40)",           lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])
    print(f"\n  [{label}]")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset: continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"    {name:<22}: n={n}  acc={acc*100:.1f}%  expected={mid*100:.0f}%  gap={gap:.3f}  {flag}")
        calib_out.append({"bucket": name, "count": n, "accuracy": round(acc,4),
                          "expected": mid, "gap": round(gap,4)})
    hc     = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"    ECE={ece:.4f}  |  High-conf: {len(hc)}  |  Acc@high={hc_acc:.1f}%  |  False-conf: {false_conf}")
    return ece, calib_out, false_conf

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION")
print("=" * 65)
c_ece, c_calib, c_false = calibration_report(cot_results,  "CoT")
b_ece, b_calib, b_false = calibration_report(base_results, "Baseline")

improve = (b_ece - c_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE: CoT={c_ece:.4f}  Baseline={b_ece:.4f}  Improvement={improve:.1f}%")
print(f"  False confidence: CoT={c_false}  Baseline={b_false}")

angle3 = {
    "cot_ece": round(c_ece, 4), "baseline_ece": round(b_ece, 4),
    "ece_improvement_pct": round(improve, 2),
    "cot_false_conf": c_false, "baseline_false_conf": b_false,
    "false_conf_reduction": b_false - c_false,
    "cot_calibration": c_calib, "baseline_calibration": b_calib,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(angle3, f, indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")

In [ ]:
# CELL 15 -- ANGLE 4: ACCURACY BY OPERATION TYPE

c_by_op = defaultdict(list)
b_by_op = defaultdict(list)
for r in cot_results:  c_by_op[r.get("op_type", "Unknown")].append(r)
for r in base_results: b_by_op[r.get("op_type", "Unknown")].append(r)
all_ops = sorted(set(list(c_by_op.keys()) + list(b_by_op.keys())))

print("=" * 65)
print("ANGLE 4 -- BY OPERATION TYPE")
print("=" * 65)
print(f"\n  {'Op':<16} | {'N':>4} | {'CoT':>7} | {'Base':>7} | {'Gain':>6}")
op_results = {}
for op in all_ops:
    c_items = c_by_op.get(op, [])
    b_items = b_by_op.get(op, [])
    n = len(c_items)
    if n == 0: continue
    c_acc = sum(r["correct"] for r in c_items) / n * 100
    b_acc = sum(r["correct"] for r in b_items) / max(len(b_items), 1) * 100
    gain  = c_acc - b_acc
    print(f"  {op:<16} | {n:>4} | {c_acc:>6.1f}% | {b_acc:>6.1f}% | {gain:>+6.1f}%")
    op_results[op] = {
        "n": n, "cot_acc": round(c_acc,2), "baseline_acc": round(b_acc,2),
        "gain": round(gain,2),
    }

print("\n  Ranked by CoT gain:")
for op, v in sorted(op_results.items(), key=lambda x: -x[1]["gain"]):
    print(f"    {op:<16}: {v['gain']:>+.1f}%")

angle4 = {"dataset": "ASDiv", "by_operation_type": op_results}
with open(CONFIG["angle4_file"], "w") as f:
    json.dump(angle4, f, indent=2)
print(f"\nSaved -> {CONFIG['angle4_file']}")

In [ ]:
# CELL 16 -- 3-WAY COMPARISON SUMMARY (Paper Table)
# Paste your Guided results from the original notebook below.

# ---- Paste guided results from original notebook ----
GUIDED_ACC     = 64.0   # guided accuracy %
GUIDED_ECE     = 0.100  # guided ECE
GUIDED_COMPUTE = 10.5   # B param-passes
# -----------------------------------------------------

c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
c_ece = json.load(open(CONFIG["angle3_file"]))["cot_ece"]
b_ece = json.load(open(CONFIG["angle3_file"]))["baseline_ece"]

print("=" * 70)
print("  ASDIV -- 3-WAY COMPARISON  (Paper Table)")
print("=" * 70)
print(f"  N={len(cot_results)}  Seed={CONFIG['random_seed']}  Solver=Qwen2.5-1.5B")
print()
print(f"  {'Condition':<30} | {'Compute':>7} | {'Accuracy':>9} | {'ECE':>7} | {'vs Baseline':>12}")
print(f"  {'-'*30}-+-{'-'*7}-+-{'-'*9}-+-{'-'*7}-+-{'-'*12}")
print(f"  {'Baseline (no guide, no CoT)':<30} | {'7.5B':>7} | {b_acc:>8.1f}% | {b_ece:>7.4f} | {'—':>12}")
print(f"  {'CoT (think step by step)':<30} | {'7.5B':>7} | {c_acc:>8.1f}% | {c_ece:>7.4f} | {c_acc-b_acc:>+11.1f}%")
print(f"  {'Guided (fine-tuned 3B guide)':<30} | {'10.5B':>7} | {GUIDED_ACC:>8.1f}% | {GUIDED_ECE:>7.4f} | {GUIDED_ACC-b_acc:>+11.1f}%")
print()
print(f"  KEY QUESTION: Does Guided beat CoT?")
print(f"  Guided vs CoT: {GUIDED_ACC - c_acc:+.1f} pts")
if GUIDED_ACC > c_acc + 2:
    print("  RESULT: Guided clearly outperforms CoT. Guide model justified.")
elif abs(GUIDED_ACC - c_acc) <= 2:
    print("  RESULT: Marginal difference (<2pts). Report honestly -- discuss tradeoff.")
else:
    print("  RESULT: CoT matches/beats Guided. Important finding to report honestly.")